## Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from xgboost import XGBClassifier
from ucimlrepo import fetch_ucirepo
import warnings
warnings.filterwarnings('ignore')

---
# PART 1: BINARY CLASSIFICATION WITH SVM (30 POINTS)
---

## Load the Sonar Dataset from UCI Repository

In [2]:
print("Loading Sonar dataset...")
sonar = fetch_ucirepo(id=151)
X_sonar = sonar.data.features
y_sonar = sonar.data.targets

y_sonar


Loading Sonar dataset...


,class
0,R
1,R
2,R
3,R
4,R
...,...
203,M
204,M
205,M
206,M


## Encode the Target Labels using LabelEncoder()

In [3]:
# Encode labels 
encoder = LabelEncoder()
y_transformed = encoder.fit_transform(y_sonar) 
y_transformed

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

## Split the Data into 80% Training and 20% Testing Sets

In [4]:
#Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X_sonar, y_transformed, 
                                                    test_size=0.2, 
                                                    random_state=50, 
                                                    stratify=y_transformed)

---
## 1.1. Linear Kernel SVM without Hyperparameter Tuning (5 points)
---

## Create a Pipeline with StandardScaler and Linear SVM

In [5]:
# Create a pipeline using a linear 
linear_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear'))  
])

## Train the Linear SVM Model

In [6]:
linear_pipeline.fit(X_train, y_train)

,steps,"[('scaler', ...), ('svm', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'


## Predict on the Test Set and Calculate Accuracies and Print it

In [7]:
# Predict on train and test data
y_train_prediction = linear_pipeline.predict(X_train)
y_test_prediction = linear_pipeline.predict(X_test)

# Calculate Accuracies
train_accuracy = accuracy_score(y_train, y_train_prediction)
test_accuracy = accuracy_score(y_test, y_test_prediction)

# Print 
print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)

Train Accuracy: 0.9578313253012049
Test Accuracy: 0.7619047619047619


## Display Classification Report for Linear SVM

In [8]:
class_report = classification_report(y_test, y_test_prediction) # Make classification report
print("Classification report: ",class_report) # Print

Classification report:                precision    recall  f1-score   support

           0       0.77      0.77      0.77        22
           1       0.75      0.75      0.75        20

    accuracy                           0.76        42
   macro avg       0.76      0.76      0.76        42
weighted avg       0.76      0.76      0.76        42



## Display Confusion Matrix for Linear SVM

In [9]:
conf_matrix = confusion_matrix(y_test, y_test_prediction) # Make confusion matrix
print(conf_matrix) # Print

[[17  5]
 [ 5 15]]


---
## 1.2. SVM with GridSearchCV and 5-Fold Cross-Validation (15 points)
---

## Create a Pipeline for GridSearchCV

In [10]:
# Create pipeline
grid_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC()) 
])

## Define the Parameter Grid for Different Kernels

In [11]:
parameter_grid = [
    # Linear kernel
    {
        'svm__kernel': ['linear'],
        'svm__C': [0.1, 1, 10, 100]
    },

    # RBF kernel
    {
        'svm__kernel': ['rbf'],
        'svm__C': [0.1, 1, 10, 100],
        'svm__gamma': ['scale', 'auto', 0.001, 0.01]
    },

    # Polynomial kernel
    {
        'svm__kernel': ['poly'],
        'svm__C': [0.1, 1, 10, 100],
        'svm__gamma': ['scale', 'auto', 0.001, 0.01],
        'svm__degree': [2, 3, 4]
    }
]

## Create Stratified 5-Fold Cross-Validation

In [12]:
# StratifiedKFold with 5 splits
cross_val = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

## Initialize and Run GridSearchCV on Training Data

In [13]:

# Initialize
grid_svm = GridSearchCV(estimator=grid_pipeline, param_grid=parameter_grid, cv=cross_val, scoring='accuracy')

# Train GridSearchCV
grid_svm.fit(X_train, y_train)

,estimator,"Pipeline(step...svm', SVC())])"
,param_grid,"[{'svm__C': [0.1, 1, ...], 'svm__kernel': ['linear']}, {'svm__C': [0.1, 1, ...], 'svm__gamma': ['scale', 'auto', ...], 'svm__kernel': ['rbf']}, ...]"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


## Display Best Parameters and Cross-Validation Score

In [14]:

# Take best parameters and best score
best_parameters = grid_svm.best_params_
best_score = grid_svm.best_score_

# Print best parameters and best score
print("Best Parameters:", best_parameters)
print("Best Cross-Validation Score:", best_score)

Best Parameters: {'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Best Cross-Validation Score: 0.8789661319073083


## Evaluate the Best Model on the Test Set

In [15]:
# Take best model
best_model = grid_svm.best_estimator_ 

# Predict
y_pred_best_model = best_model.predict(X_test)

# Calculate Accuracy
test_acc_best_model = accuracy_score(y_test, y_pred_best_model)

# Print accuracu
print("Test Accuracy:", test_acc_best_model)

Test Accuracy: 0.8809523809523809


## Display Top 5 Parameter Combinations

In [16]:
# take results and put into dict
cv_results =  grid_svm.cv_results_

# get ranks
ranks = cv_results['rank_test_score']

# Get the indices that would sort the ranks 
sorted_indices = np.argsort(ranks)

# Get the indices of top 5 
top_5_indices = sorted_indices[:5]


# Loop and print
for i in top_5_indices:
    rank = cv_results['rank_test_score'][i]
    score = cv_results['mean_test_score'][i]
    params = cv_results['params'][i]

    print(rank, score , params)

1 0.8789661319073083 {'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
1 0.8789661319073083 {'svm__C': 10, 'svm__gamma': 'auto', 'svm__kernel': 'rbf'}
1 0.8789661319073083 {'svm__C': 100, 'svm__gamma': 'auto', 'svm__kernel': 'rbf'}
1 0.8789661319073083 {'svm__C': 100, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
5 0.8729055258467022 {'svm__C': 10, 'svm__gamma': 0.01, 'svm__kernel': 'rbf'}


## Display Classification Report for Tuned SVM

In [17]:
class_report = classification_report(y_test, y_pred_best_model) # Make classification report
print("Classification report: ",class_report) # Print

Classification report:                precision    recall  f1-score   support

           0       0.84      0.95      0.89        22
           1       0.94      0.80      0.86        20

    accuracy                           0.88        42
   macro avg       0.89      0.88      0.88        42
weighted avg       0.89      0.88      0.88        42



## Display Confusion Matrix for Tuned SVM

In [18]:
conf_matrix = confusion_matrix(y_test, y_pred_best_model) # Make confusion matrix
print("Confusion matrix: ",conf_matrix) # Print

Confusion matrix:  [[21  1]
 [ 4 16]]


## Compare Linear SVM vs Tuned SVM Results. Compare experimental results. Explain the performance impact of linear and nonlinear kernels. Why is kernel trick important? Also explain why using k-fold cross validation is more advantageous than train-test split. (10 points)

With Linear SVM, we achieved 95% train accuracy and 76% test accuracy. This result is normal given the complexity of the dataset and the Linear SVM model we used. When we used Tuned SVM, GridSearchCV selected the best model and hyperparameters for the dataset, resulting in 88% test accuracy.

The linear kernel didn't perform very well. This is because it only performs well when the data is linearly separable, and because our dataset wasn't well-suited to the linear kernel, it didn't perform as well as the nonlinear kernel. However, the nonlinear kernel performed very well because nonlinear kernels project the data into a higher-dimensional space, making it linearly separable there. They create flexible, curved decision boundaries. The RBF kernel was the most suitable kernel for the dataset we used.

Kernel trick is important because the kernel trick allows us to classify nonlinear data using a linear classifier. This method was very useful for our dataset. The kernel trick increased the model's performance while significantly reducing computational costs thanks to the dot product calculation.

First, K-fold cross-validation is more robust than the train-test split because the train-test split is split only once, and accuracy depends on which data points fall into the test set by chance. In K-fold cross-validation, the test is performed k different times and averaged over each test, providing a much more reliable estimate of the model's true performance. K-fold cross-validation is more efficient than using the train-test split because K-fold allows each data point to be used for both training and validation. This is quite efficient for small data sets like ours.

---
# PART 2: MULTICLASS CLASSIFICATION (70 POINTS)
---

## Load the Dry Beans Dataset from UCI Repository

In [19]:
print("Loading Dry Beans dataset...")
dry_bean = fetch_ucirepo(id=602)
X_beans = dry_bean.data.features
y_beans = dry_bean.data.targets
y_beans

Loading Dry Beans dataset...


,Class
0,SEKER
1,SEKER
2,SEKER
3,SEKER
4,SEKER
...,...
13606,DERMASON
13607,DERMASON
13608,DERMASON
13609,DERMASON


## Encode the Target Labels using LabelEncoder

In [20]:
# Encode labels 
encoder = LabelEncoder()
y_transformed = encoder.fit_transform(y_beans) 
y_transformed

array([5, 5, 5, ..., 3, 3, 3], shape=(13611,))

## Split the Data into 80% Training and 20% Testing Sets

In [21]:
# Split the data 
X_train, X_test, y_train, y_test = train_test_split(X_beans, y_transformed, test_size=0.2, random_state=50, stratify=y_transformed)

## Scale the Features using StandardScaler

In [22]:
# Standardize features 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

---
## 2.1. Multinomial Logistic Regression (20 points)
---

## Define the Multinomial Logistic Regression Class

In [23]:
class MultinomialLogisticRegression:
    def __init__(self, learning_rate=0.01, epochs=1000, reg_lambda=0.01, random_state=50):
        # hyperparameters
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.reg_lambda = reg_lambda
        self.random_state = random_state

        # Model parameters
        self.weights = None
        self.bias = None

    def _softmax(self,X):
        # Compute raw scores
        scores = X.dot(self.weights) + self.bias

        # Softmax numerators
        numerator = np.exp(scores) 

        # Softmax denominators
        denum = np.sum(numerator, axis=1, keepdims=True)

        return numerator/denum
    
    def _cross_entropy(self, y_true_one_hot, X):

        num_of_samples = y_true_one_hot.shape[0]

        # Predicted softmax probabilities
        y_prob = self._softmax(X)

        # Cross-entropy loss
        loss = -np.sum(y_true_one_hot * np.log(y_prob)) / num_of_samples

        # L2 regularization term
        w_square = self.weights* self.weights
        penalty = (self.reg_lambda / 2) * np.sum(w_square)

        # Total loss
        total_loss = loss + penalty
        return total_loss
    


    def predict_proba(self, X):
        return self._softmax(X) # Return softmax probabilities

    def predict(self, X):
        # Get class probabilities
        probabilities = self.predict_proba(X)
        n_samples = probabilities.shape[0]
        predictions = np.zeros(n_samples, dtype=int)
        
        # Find max per sapmle
        for i in range(n_samples):
            current_row_probs = probabilities[i]
            max_prob = -1.0
            max_index = 0
            
            for j in range(len(current_row_probs)):
                prob = current_row_probs[j]
                if prob > max_prob:
                    max_prob = prob
                    max_index = j
            
            predictions[i] = max_index
            
        return predictions


    def fit(self, X, y):
        # Dataset dimensions
        n_samples, n_features = X.shape
        n_classes = 7

        # One-hot encoding of labels
        y_one_hot = np.zeros((n_samples, n_classes))
        for i in range(n_samples):
            correct_class_index = y[i]
            y_one_hot[i, correct_class_index] = 1

        # Initialize parameters
        self.weights = np.random.randn(n_features, n_classes) * 0.01
        self.bias = np.zeros(n_classes)


        # Gradient descent loop
        for epoch in range(self.epochs):
            
            y_prob_for_gradient = self._softmax(X)
           
           # Error term
            error = (y_prob_for_gradient - y_one_hot) / n_samples
            
            # Weight update (with L2 regularization)
            self.weights = self.weights - self.learning_rate * (X.T.dot(error) + self.reg_lambda * self.weights)

            # Bias update
            self.bias = self.bias - self.learning_rate * (np.sum(error, axis=0))
        
        return self
   

        
    def score(self, X, y):
        # Compute accuracy score
        y_pred = self.predict(X)
        acc = accuracy_score(y, y_pred)
        return acc
    
    
    def get_params(self, deep=True):
        # Get parameters
        return {
            "learning_rate": self.learning_rate,
            "epochs": self.epochs,
            "reg_lambda": self.reg_lambda,
            "random_state": self.random_state,
        }

        
    def set_params(self,learning_rate=None,epochs=None,reg_lambda=None,random_state = None):
        # Update parameters
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.reg_lambda = reg_lambda
        self.random_state = random_state

        return self
    


## Define Hyperparameter Grid for Multinomial Logistic Regression

In [24]:
# Define hyperparameter
param_grid_mlr = {
    'learning_rate': [0.01, 0.05, 0.1],
    'epochs': [200, 300, 500],
    'reg_lambda': [0.0, 0.01, 0.1]  
}

# Get Multinomial Logistic Regression model and StratifiedKFold for gridsearchcv
decision_MLR_model = MultinomialLogisticRegression(random_state=50)
cross_val_MLR = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)

## Run GridSearchCV for Multinomial Logistic Regression

In [25]:
# Build GridSearchCV
grid_search_MLR = GridSearchCV(estimator=decision_MLR_model, param_grid=param_grid_mlr, cv=cross_val_MLR, scoring='accuracy')

# Train GridSearchCV
grid_search_MLR.fit(X_train_scaled, y_train)


# Take best parameters and best score
best_parameters_MLR = grid_search_MLR.best_params_
best_score_MLR = grid_search_MLR.best_score_

# Print best parameters and best score
print("Best Parameters:", best_parameters_MLR)
print("Best Cross-Validation Score:", best_score_MLR)

Best Parameters: {'epochs': 500, 'learning_rate': 0.1, 'reg_lambda': 0.0}
Best Cross-Validation Score: 0.9175237572197525


## Evaluate Multinomial Logistic Regression on Test Set

In [26]:
# Take best model
best_model_MLR = grid_search_MLR.best_estimator_ 

# Predict
y_pred_best_model_MLR = best_model_MLR.predict(X_test_scaled)

# Calculate Accuracy
test_acc_best_model_MLR = accuracy_score(y_test, y_pred_best_model_MLR)

# Print accuracy
print("Test Accuracy:", test_acc_best_model_MLR)

Test Accuracy: 0.914799853103195


## Display Classification Report for Multinomial Logistic Regression

In [27]:
class_report = classification_report(y_test, y_pred_best_model_MLR) # Make classification report
print("Classification report: ",class_report) # Print

Classification report:                precision    recall  f1-score   support

           0       0.94      0.84      0.89       265
           1       1.00      0.98      0.99       104
           2       0.88      0.96      0.92       326
           3       0.90      0.93      0.91       709
           4       0.95      0.95      0.95       386
           5       0.95      0.95      0.95       406
           6       0.87      0.84      0.86       527

    accuracy                           0.91      2723
   macro avg       0.93      0.92      0.92      2723
weighted avg       0.92      0.91      0.91      2723



## Display Confusion Matrix for Multinomial Logistic Regression

In [28]:
conf_matrix = confusion_matrix(y_test, y_pred_best_model_MLR) # Make confusion matrix
print("Confusion matrix: ",conf_matrix) # Print

Confusion matrix:  [[223   0  28   0   1   2  11]
 [  0 102   2   0   0   0   0]
 [  4   0 312   0   6   1   3]
 [  1   0   0 659   1   9  39]
 [  1   0  11   2 367   0   5]
 [  6   0   0  10   0 384   6]
 [  1   0   0  63  12   7 444]]


---
## 2.2. Decision Tree (20 points)
---

###  Define the Decision Tree model. Use GridSearchCV to find the best hyperparameters. Perform multiclass classification on the Dry Beans dataset. Print best hyperparameters, classification report and confusion matrix. (15 points)

## Define Hyperparameter Grid for Decision Tree

In [29]:
# Define hyperparameter
param_grid_decision_t = {
    'max_depth': [10, 15, 20, 25, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

# Get DecisionTree model and StratifiedKFold for gridsearchcv
decision_t_model = DecisionTreeClassifier(random_state=50)
cross_val_dt = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)

## Run GridSearchCV for Decision Tree

In [30]:
# Build GridSearchCV
grid_search_decision_t = GridSearchCV(estimator=decision_t_model, param_grid=param_grid_decision_t, cv=cross_val_dt, scoring='accuracy')

# Train GridSearchCV
grid_search_decision_t.fit(X_train_scaled, y_train)


# Take best parameters and best score
best_parameters_decision_t = grid_search_decision_t.best_params_
best_score_decision_t = grid_search_decision_t.best_score_

# Print best parameters and best score
print("Best Parameters:", best_parameters_decision_t)
print("Best Cross-Validation Score:", best_score_decision_t)


Best Parameters: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 10}
Best Cross-Validation Score: 0.904757517970029


In [31]:
# Take best model
best_model_dt = grid_search_decision_t.best_estimator_ 

# Predict
y_pred_best_model_dt = best_model_dt.predict(X_test_scaled)

# Calculate Accuracy
test_acc_best_model_dt = accuracy_score(y_test, y_pred_best_model_dt)

# Print accuracy
print("Test Accuracy:", test_acc_best_model_dt)

Test Accuracy: 0.9107601909658465


## Display Classification Report for Decision Tree

In [32]:
class_report = classification_report(y_test, y_pred_best_model_dt) # Make classification report
print("Classification report: ",class_report) # Print

Classification report:                precision    recall  f1-score   support

           0       0.91      0.86      0.88       265
           1       1.00      0.99      1.00       104
           2       0.90      0.95      0.92       326
           3       0.91      0.92      0.91       709
           4       0.94      0.93      0.94       386
           5       0.95      0.93      0.94       406
           6       0.85      0.86      0.85       527

    accuracy                           0.91      2723
   macro avg       0.92      0.92      0.92      2723
weighted avg       0.91      0.91      0.91      2723



## Display Confusion Matrix for Decision Tree

In [33]:
conf_matrix = confusion_matrix(y_test, y_pred_best_model_dt) # Make confusion matrix
print("Confusion matrix: ",conf_matrix) # Print

Confusion matrix:  [[227   0  26   1   1   3   7]
 [  1 103   0   0   0   0   0]
 [  9   0 309   0   5   1   2]
 [  0   0   0 651   0  12  46]
 [  3   0   9   1 360   0  13]
 [  6   0   0  11   1 379   9]
 [  3   0   1  50  17   5 451]]


###  Describe how a decision tree builds its decision structure. (5 points)

The Decision Tree Algorithm creates a tree structure where each internal node represents a feature test, each branch represents an outcome, and each leaf represents a class label. The build process operates recursively. The algorithm begins at the root node, which contains the entire training dataset. The goal is to find the best possible question that will separate the dataset into two purer subgroups. This question is determined by calculating the entropy and Gini impurity. A lower value for both metrics indicates purer data. For a single node, all features and threshold values ​​for those values ​​are tested, and the entropy is calculated separately for each. The lowest entropy indicates the highest purity, and these values ​​are selected. This question generates new branches, creating new nodes. We treat these new nodes as root nodes and allow the tree to grow. This branching continues as long as a stopping criterion is not met.

The tree's growth stops when a node becomes perfectly pure or when predetermined parameters, such as max_depth or min_samples_split, are reached.

---
## 2.3. XGBoost (20 points)
---

###  Define the XGBoost model. Use GridSearchCV to find the best hyperparameters. Perform multiclass classification on the Dry Beans dataset. Print best hyperparameters, classification report and confusion matrix(10 points)

## Define Hyperparameter Grid for XGBoost

In [34]:
# Define hyperparameter
param_grid_xgb = {
    'max_depth': [3, 5, 7, 10 ],           
    'learning_rate': [0.01, 0.1, 0.3],     
    'n_estimators': [100, 200, 300],       
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Get XGBoost model and StratifiedKFold for gridsearchcv
xgb_model = XGBClassifier(
    eval_metric='mlogloss',     # Evaluation metric as specified
    random_state=50, 
)

cross_val_xgb = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Run GridSearchCV for XGBoost

In [35]:
# Build GridSearchCV
grid_search_XGB = GridSearchCV(estimator=xgb_model, param_grid=param_grid_xgb, cv=cross_val_xgb, scoring='accuracy')

# Train GridSearchCV
grid_search_XGB.fit(X_train_scaled, y_train)


# Take best parameters and best score
best_parameters_XGB = grid_search_XGB.best_params_
best_score_XGB = grid_search_XGB.best_score_

# Print best parameters and best score
print("Best Parameters:", best_parameters_XGB)
print("Best Cross-Validation Score:", best_score_XGB)

Best Parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}
Best Cross-Validation Score: 0.9287282563809894


## Evaluate XGBoost on Test Set

In [36]:
# Take best model
best_model_XGB = grid_search_XGB.best_estimator_ 

# Predict
y_pred_best_model_XGB = best_model_XGB.predict(X_test_scaled)

# Calculate Accuracy
test_acc_best_model_XGB = accuracy_score(y_test, y_pred_best_model_XGB)

# Print accuracy
print("Test Accuracy:", test_acc_best_model_XGB)

Test Accuracy: 0.9408740359897172


## Display Classification Report for XGBoost

In [37]:
class_report = classification_report(y_test, y_pred_best_model_XGB) # Make classification report
print("Classification report: ",class_report) # Print

Classification report:                precision    recall  f1-score   support

           0       0.95      0.92      0.94       265
           1       0.99      1.00      1.00       104
           2       0.95      0.96      0.95       326
           3       0.92      0.95      0.93       709
           4       0.98      0.95      0.96       386
           5       0.97      0.96      0.97       406
           6       0.90      0.89      0.90       527

    accuracy                           0.94      2723
   macro avg       0.95      0.95      0.95      2723
weighted avg       0.94      0.94      0.94      2723



## Display Confusion Matrix for XGBoost

In [38]:
conf_matrix = confusion_matrix(y_test, y_pred_best_model_XGB) # Make confusion matrix
print("Confusion matrix: ",conf_matrix) # Print

Confusion matrix:  [[244   0  12   1   0   2   6]
 [  0 104   0   0   0   0   0]
 [  6   1 313   0   3   1   2]
 [  0   0   0 672   0   5  32]
 [  1   0   6   3 368   0   8]
 [  3   0   0   9   0 391   3]
 [  2   0   0  45   6   4 470]]


### What are the revolutionary features of XGBoost compared to other tree-based models? (10 points)

Other tree-based models are more prone to overfitting, but XGBoost penalizes model complexity by using L1 (Lasso) and L2 (Ridge) regularization terms.

Most tree-based models use a first-order derivative for the loss function. XGBoost, however, uses both a gradient and a Hessian. This allows for more precise loss function calculations and allows for more precise updates.

As with all boosting models, trees are built sequentially, but XGBoosting's difference lies in its ability to perform the splitting process, the most time-consuming part, in parallel. In other words, trees are still built sequentially, but it performs the best split in parallel, utilizing all CPU cores, significantly speeding up the process.

Sparsity-Awareness: XGBoost natively handles NaN data, eliminating the need for data imputation. When searching for a split on a node, XGBoost performs the following: When a NaN value is encountered, it tries both directions and chooses the direction that increases the score more. This eliminates time-consuming data imputation and saves us time.



###  Compare the classification results of your Multinomial Logistic Regression, Decision Tree, and XGBoost models on the Dry Beans dataset. Discuss the comparison in terms of overall model performance, risk of overfitting, model complexity and the scenarios in which each model is most effective. (10 points)

XGBoost delivered the best performance with 94.1% accuracy, followed by MultinomialLogisticRegression with 91.5% accuracy, and Decision Tree delivered the worst with 91.1% accuracy.

Thanks to the GridSearchCV we used, the risk of overfitting was prevented in all our models. In the Multinomial Logistic Regression class, a value of 0.0 in the best parameter, reg_lambda, indicates that the model generalizes very well without needing a penalty. In the Decision Tree, the risk of overfitting is reduced thanks to parameters such as GridSearchCV's max_depth and min_samples_split. In XGBoost, the model's inherent L1/L2 regularization significantly reduced the risk of overfitting, and the GridSearchCV max_depth parameter also contributed to this. Accordingly, the decision tree has the highest risk of overfitting because an unconstrained decision tree branches until it has memorized the data. After that, Multinomial Logistic Regression has the highest overfitting risk because, as a linear model, it has little tendency to memorize noise or outliers in the data. The model with the least overfitting risk is XGBoost, because, as mentioned above, it makes the model with the least overfitting risk thanks to its built-in regularization values ​​and other mechanisms.

Regarding model complexity, the least complex model is definitely Multinomial Logistic Regression. This is because the model is linear and has a simple structure. The second most complex model is the decision tree, as it doesn't have advanced mechanisms like XGBoost. It's not a linear model like logistic regression. The most complex model is the XGBoost model we used, because it trains all trees sequentially, and with n_estimators: 100 parameters, this model is the sum of 100 different decision trees. The combination of these many trees makes this model quite complex.

Multinominal Logistic Regression is most effective in scenarios where the data set is linearly separable. Decison Tree is most effective when the data is small or medium-sized and complex, and when the model's decisions need to be explained or interpreted. XGBoost is most effective when the data set has a complex class distribution and many features, and when the goal is to achieve the highest possible accuracy and performance.

